# Part 2: GitHub Contributor Analytics Pipeline

## Overview

| Property  | Value  |
|-----------|--------|
| Tasks     | 3      |
| Languages | Python |

### Objective
Build an ingestion and transformation pipeline using GitHub's REST API to analyze repository contributors.

### Data Source
- **Target Repository:** `apache/airflow`
- **Base URL:** `https://api.github.com`

### Endpoints
- `/repos/apache/airflow/commits`
- `/repos/apache/airflow/pulls`
- `/repos/apache/airflow/pulls/comments`
- `/repos/apache/airflow/issues`
- `/repos/apache/airflow/pulls/{pull_number}/reviews`

---
## Setup

Load environment variables and establish Snowflake connection.

In [ ]:
import os
from dotenv import load_dotenv
import snowflake.connector

load_dotenv()

assert os.environ.get('GITHUB_TOKEN'), 'GITHUB_TOKEN not set. Check .env file.'
print('Environment loaded successfully.')

# Snowflake connection for data storage
from src.docs_pipeline.config import get_snowflake_params
conn = snowflake.connector.connect(**get_snowflake_params())
print('Snowflake connected.')

---
## Create Snowflake Tables

Drop existing tables (clean slate) and recreate them.

In [ ]:
from src.github_pipeline.ddl import get_ddl, apply_ddl, drop_tables

# Drop existing tables for a clean slate
drop_tables(conn)

# Preview DDL
print(get_ddl())

# Recreate tables
apply_ddl(conn)

---
## Task 1: Data Ingestion

### Requirements
- Ingest from all endpoints listed above
- Store each endpoint as a separate dataset
- Output: Row count per endpoint after ingestion completes

Data is stored in Snowflake and supports resume. If interrupted, re-running
this cell will continue from the last checkpoint.

In [4]:
from src.github_pipeline.ingestion import run_ingestion

# Pass conn for Snowflake storage + resume; omit for fetch-only mode
data = run_ingestion(conn=conn)

print('\n--- Fetched Row Counts ---')
for key, df in data.items():
    print(f'  {key}: {len(df)} rows')

  Reviews progress: 450/500 PRs
  Reviews progress: 500/500 PRs
  Reviews: 1765 rows
  Loaded 1765 reviews into CANDIDATE_NS_GH_RAW_REVIEWS

--- Snowflake Row Counts ---
  GH_RAW_COMMITS: 1000 rows
  GH_RAW_PULLS: 1000 rows
  GH_RAW_PR_COMMENTS: 1000 rows
  GH_RAW_ISSUES: 1000 rows
  GH_RAW_REVIEWS: 1765 rows

--- Fetched Row Counts ---
  commits: 1000 rows
  pulls: 1000 rows
  pr_comments: 1000 rows
  issues: 1000 rows
  reviews: 1765 rows


---
## Task 2: Transformation

### Requirements
Build a contributor analytics dataset by joining ingested data.

### Output Schema

| Column       | Description                            |
|-------------|----------------------------------------|
| author      | GitHub username (login)                 |
| commits     | Commit count                            |
| prs         | Pull request count (as author)          |
| comments    | PR comment count                        |
| reviews     | Review count                            |
| score       | Weighted score (formula below), max 100 |
| tier        | Classification based on activity        |
| overall_rank| Global rank by score (1 = highest)      |
| tier_rank   | Rank within tier                        |
| percentile  | Score percentile (0-100 scale)          |

### Scoring Formula
```
raw_score = (commits x 5) + (prs x 10) + (comments x 2) + (reviews x 3)
score = min(raw_score, 100)
```

### Tier Definitions

| Tier        | Criteria              |
|------------|----------------------|
| core       | commits + prs >= 20   |
| active     | commits + prs >= 5    |
| contributor| commits + prs >= 1    |
| observer   | commits + prs = 0     |

In [5]:
from src.github_pipeline.transformation import process_contributors, print_summary

contributors_df = process_contributors(data)
print(f'Contributors DataFrame: {contributors_df.shape[0]} rows, {contributors_df.shape[1]} columns')
contributors_df.head(10)

Contributors DataFrame: 435 rows, 10 columns


,author,commits,prs,comments,reviews,score,tier,overall_rank,tier_rank,percentile
0,Henry Chen,24,0,0,0,100,core,1,1,94.37
1,github-actions[bot],0,110,0,0,100,core,1,1,94.37
2,dheerajturaga,0,18,22,10,100,active,1,1,94.37
3,Yeonguk Choo,23,0,0,0,100,core,1,1,94.37
4,eladkal,0,13,8,33,100,active,1,1,94.37
5,vatsrahul1001,0,12,6,30,100,active,1,1,94.37
6,Prab-27,0,8,9,8,100,active,1,1,94.37
7,bugraoz93,0,27,24,62,100,core,1,1,94.37
8,ashb,0,1,6,53,100,contributor,1,1,94.37
9,Amogh Desai,60,0,0,0,100,core,1,1,94.37


### Required Output

In [6]:
print_summary(contributors_df)

Top 10 Contributors by Score:
                author  score         tier
0           Henry Chen    100         core
1  github-actions[bot]    100         core
2        dheerajturaga    100       active
3         Yeonguk Choo    100         core
4              eladkal    100       active
5        vatsrahul1001    100       active
6              Prab-27    100       active
7            bugraoz93    100         core
8                 ashb    100  contributor
9          Amogh Desai    100         core

Tier Distribution:
tier
contributor    322
active          77
core            22
observer        14
Name: count, dtype: int64

Summary:
Total contributors: 435
Min score: 3
Max score: 100
Count achieving max score (100): 50


---
## Task 3: Export to Google Sheets

### Requirements
- Export contributor analytics to Google Sheets
- Uses `GITHUB_SHEETS_SPREADSHEET_ID` and `GOOGLE_APPLICATION_CREDENTIALS` from `.env`

In [8]:
from src.github_pipeline.export_sheets import export_to_sheets
from src.github_pipeline.config import get_sheets_spreadsheet_id

spreadsheet_id = '1Wcy2UrqUk-MzCZpsgPWq9QWzrLxZgxTTADj1z-dItjw'
result = export_to_sheets(contributors_df, spreadsheet_id=spreadsheet_id)

print(f"\nExport complete!")
print(f"View your sheet at: https://docs.google.com/spreadsheets/d/{result['spreadsheet_id']}/edit")

Using Google Service Account: sheets-snowflae-access@firegram-1r.iam.gserviceaccount.com
Created sheet 'Part2_Contributors'
Wrote 436 rows to 'Part2_Contributors'

Export complete!
View your sheet at: https://docs.google.com/spreadsheets/d/1Wcy2UrqUk-MzCZpsgPWq9QWzrLxZgxTTADj1z-dItjw/edit


---
## Cleanup

In [ ]:
conn.close()
print('Snowflake connection closed.')